
# 🔢 NB03 — MapReduce + Word Count
## Investor Intelligence Platform — FIIs Brasil 🇧🇷
### CRISP-DM Phase: Modeling

| | |
|---|---|
| **Input** | `data/silver/silver_articles.parquet` |
| **Output** | `data/gold/word_count/*.parquet` (4 artefatos) |
| **RDD ops** | `parallelize` → `flatMap` → `map` → `reduceByKey` → `sortBy` → `collect` |

## 🎯 O que este notebook faz

1. **MapReduce Global** — frequência de cada termo no corpus das 21 fontes.
2. **Word Count por fonte** — top termos por portal.
3. **TOFU Term Frequency** — termos de funil de marketing rankeados.
4. **Negative Context** — termos de alta frequência em contexto negativo.

> 🎓 O pattern `flatMap → map → reduceByKey` é o MapReduce canônico distribuído.
> A mesma lógica que processa 1 M de documentos em cluster funciona aqui — só muda o número de executores.

## 🏛️ Arquitetura MapReduce

```
Silver Corpus (N artigos, M tokens)
        │
   parallelize(texts)
        │
   ┌────▼─────────────┐
   │   MAP PHASE      │
   │  flatMap(tokens) │  → cada texto → lista de tokens
   │  map(w → (w,1))  │  → cada token → (word, 1)
   └────┬─────────────┘
        │  Shuffle (by key)
   ┌────▼─────────────┐
   │  REDUCE PHASE    │
   │  reduceByKey(+)  │  → soma por palavra
   └────┬─────────────┘
        │
   sortBy(-count) → collect()
        │
   data/gold/word_count/
   ├── global_word_count.parquet
   ├── source_word_count.parquet
   ├── tofu_frequency.parquet
   └── negative_context.parquet
```

## 📦 Seção 1 — Imports

In [1]:

import sys, os
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
import re, random, warnings, unicodedata
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import nltk

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

warnings.filterwarnings("ignore")
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"NLTK    : {nltk.__version__}")
print(f"Pandas  : {pd.__version__}")

Python  : 3.12.12
PySpark : 4.1.2
NLTK    : 3.9.4
Pandas  : 3.0.3


## ⚙️ Seção 2 — Configuração, Paths e SparkSession

In [2]:

PROJECT_ROOT = Path(os.getcwd()).resolve()
_lim = 6
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.parent != PROJECT_ROOT and _lim > 0:
    PROJECT_ROOT = PROJECT_ROOT.parent; _lim -= 1
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import (
    RANDOM_SEED, SPARK_DRIVER_MEMORY, SPARK_APP_NAME, SPARK_SHUFFLE_PARTS,
)
from src.utils.logger import ingestion_logger, log_quality_check, log_spark_start

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SILVER_DIR  = PROJECT_ROOT / "data" / "silver"
GOLD_WC_DIR = PROJECT_ROOT / "data" / "gold" / "word_count"
GOLD_WC_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}_wc")
    .master("local[*]")
    .config("spark.driver.memory",          SPARK_DRIVER_MEMORY)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTS)
    .config("spark.sql.adaptive.enabled",   "false")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

logger = ingestion_logger()
log_spark_start(logger, spark.sparkContext.appName, spark.version)

# NLTK stopwords PT-BR
for _res in ["stopwords", "punkt", "punkt_tab"]:
    try:
        nltk.data.find(f"corpora/{_res}")
    except LookupError:
        try: nltk.download(_res, quiet=True)
        except: pass
try:
    from nltk.corpus import stopwords as _nltk_sw
    STOPWORDS_PT = set(_nltk_sw.words("portuguese"))
except Exception:
    STOPWORDS_PT = set()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"GOLD_WC_DIR  : {GOLD_WC_DIR}")
print(f"Spark        : {spark.sparkContext.appName} | {spark.version}")
print(f"Stopwords PT : {len(STOPWORDS_PT)} termos")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/14 15:46:03 WARN Utils: Your hostname, MacBook-Air-100.local, resolves to a loopback address: 127.0.0.1; using 192.168.15.4 instead (on interface en0)
26/06/14 15:46:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 15:46:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 15:46:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


2026-06-14T15:46:05 | INFO     | fii_pipeline.ingestion | SPARK_START | app=FIIIntelligencePlatform_wc | spark=4.1.2
PROJECT_ROOT : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform
GOLD_WC_DIR  : /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/word_count
Spark        : FIIIntelligencePlatform_wc | 4.1.2
Stopwords PT : 207 termos


## 📂 Seção 3 — Carregar Silver

In [3]:

SILVER_PATH = SILVER_DIR / "silver_articles.parquet"
if not SILVER_PATH.exists():
    raise FileNotFoundError(f"Silver não encontrado: {SILVER_PATH}\nExecute NB02 primeiro.")

df_silver = pd.read_parquet(SILVER_PATH)
_REQ = ["article_id", "text_clean", "source_label", "source_type", "word_count", "has_content"]
_miss = [c for c in _REQ if c not in df_silver.columns]
if _miss:
    raise ValueError(f"Colunas ausentes no Silver: {_miss}")

df_silver = df_silver[df_silver["has_content"] == True].copy()
df_silver["text_clean"] = df_silver["text_clean"].fillna("").astype(str)

print(f"Silver carregado: {len(df_silver):,} registros úteis")
print(f"\n  source_type:\n{df_silver['source_type'].value_counts().to_string()}")
print(f"\n  word_count médio : {df_silver['word_count'].mean():.1f}")
print(f"  word_count total : {df_silver['word_count'].sum():,}")

Silver carregado: 126 registros úteis

  source_type:
source_type
rss         69
scraping    55
reddit       2

  word_count médio : 593.5
  word_count total : 74,786



## 🔤 Seção 4 — Tokenizador NLP PT-BR

**Pipeline de normalização:**
```
Input: "Fundos Imobiliários pagam Dividendos! Leia mais: https://..."
1. lowercase             → "fundos imobiliários pagam dividendos!..."
2. regex [a-zà-ü]{3,}   → ["fundos","imobiliários","pagam","dividendos"]
3. NFD normalize         → ["fundos","imobiliarios","pagam","dividendos"]
4. remove stopwords      → ["fundos","imobiliarios","dividendos"]
```

**Por que NFD?** `imobiliários` e `imobiliarios` devem mapear ao **mesmo** termo.

In [4]:

_TOKEN_RE = re.compile(r"[a-zà-ü]{3,}", re.IGNORECASE | re.UNICODE)

def _norm(s: str) -> str:
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode("ascii").lower()

def tokenize(text: str) -> list:
    if not text or not isinstance(text, str): return []
    tokens = [_norm(t) for t in _TOKEN_RE.findall(text.lower())]
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) >= 3]

# Dicionário de termos TOFU (topo de funil de marketing FII)
TOFU_TERMS = [
    "fii", "fiis", "fundo", "fundos", "imobiliario", "imobiliarios",
    "dividendo", "dividendos", "provento", "proventos", "rendimento",
    "yield", "cotista", "cota", "cotas", "vacancia", "inadimplencia",
    "gestao", "portfolio", "renda", "passiva", "investimento",
    "tijolo", "papel", "logistica", "shopping", "laje", "galpao",
    "p_vp", "pvp", "liquidez", "volatilidade",
]

# Dicionário de contexto negativo FII
NEGATIVE_CONTEXT_TERMS = {
    "vacancia": "high", "inadimplencia": "high", "crise": "high",
    "risco": "medium", "queda": "medium", "reducao": "medium",
    "corte": "medium", "perda": "medium", "desvalorizacao": "medium",
    "prejuizo": "high", "calote": "high", "default": "high",
    "venda": "low", "volatilidade": "low", "queda": "medium",
    "baixo": "low", "fraco": "low", "negativo": "high",
}

print("✅ Tokenizador definido")
# Smoke test
_test = tokenize("Os fundos imobiliários pagaram dividendos recordes este trimestre!")
print(f"  Smoke test: {_test}")

✅ Tokenizador definido
  Smoke test: ['fundos', 'imobiliarios', 'pagaram', 'dividendos', 'recordes', 'trimestre']


## 🗺️ Seção 5 — MapReduce Global Word Count (RDD)

In [5]:

print("\n🗺️ MapReduce Global Word Count")
print(f"   Corpus: {len(df_silver):,} documentos")

# ── MAP PHASE: cada texto → (token, 1) ───────────────────────────────────────
texts_rdd = spark.sparkContext.parallelize(df_silver["text_clean"].tolist())

tokens_rdd = (
    texts_rdd
    .flatMap(lambda text: tokenize(text))          # → token stream
    .map(lambda w: (w, 1))                          # → (word, 1)
)

# ── REDUCE PHASE: reduceByKey soma contagens ──────────────────────────────────
word_counts_rdd = (
    tokens_rdd
    .reduceByKey(lambda a, b: a + b)               # → (word, count)
    .sortBy(lambda x: x[1], ascending=False)
    .collect()
)

df_global_wc = pd.DataFrame(word_counts_rdd, columns=["term", "count"])
df_global_wc["rank"] = range(1, len(df_global_wc) + 1)

print(f"\n  Vocabulário único: {len(df_global_wc):,} termos")
print(f"  Top 20 termos:\n{df_global_wc.head(20).to_string(index=False)}")

_path = GOLD_WC_DIR / "global_word_count.parquet"
df_global_wc.to_parquet(_path, index=False, compression="snappy")
print(f"\n✅ Salvo: {_path}")


🗺️ MapReduce Global Word Count
   Corpus: 126 documentos



  Vocabulário único: 7,752 termos
  Top 20 termos:
        term  count  rank
     mercado    289     1
         nao    237     2
       fundo    189     3
    carteira    186     4
      fundos    183     5
      imagem    183     6
       renda    183     7
      trocar    181     8
      ativos    178     9
  dividendos    170    10
   empiricus    165    11
       risco    158    12
investidores    157    13
       sobre    155    14
       acoes    153    15
       ainda    152    16
        fiis    152    17
        pode    137    18
       todos    131    19
     segundo    128    20

✅ Salvo: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/word_count/global_word_count.parquet


## 📊 Seção 6 — Word Count por Fonte (RDD)

In [6]:

print("\n📊 Word Count por Fonte")

source_texts = list(zip(df_silver["source_label"].tolist(),
                        df_silver["text_clean"].tolist()))

source_rdd = spark.sparkContext.parallelize(source_texts)

source_wc_rdd = (
    source_rdd
    .flatMap(lambda x: [((x[0], w), 1) for w in tokenize(x[1])])  # ((source, word), 1)
    .reduceByKey(lambda a, b: a + b)                               # ((source, word), count)
    .map(lambda x: (x[0][0], x[0][1], x[1]))                      # (source, word, count)
    .sortBy(lambda x: (x[0], -x[2]))
    .collect()
)

df_src_wc = pd.DataFrame(source_wc_rdd, columns=["source_label", "term", "count"])

# Top 10 termos por fonte
df_src_top = (
    df_src_wc
    .groupby("source_label")
    .apply(lambda g: g.nlargest(10, "count"))
    .reset_index(drop=True)
)

_path = GOLD_WC_DIR / "source_word_count.parquet"
df_src_wc.to_parquet(_path, index=False, compression="snappy")
print(f"  Fontes: {df_src_wc['source_label'].nunique()}")
print(f"  Pares (source, term): {len(df_src_wc):,}")
print(f"✅ Salvo: {_path}")


📊 Word Count por Fonte
  Fontes: 16
  Pares (source, term): 13,085
✅ Salvo: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/word_count/source_word_count.parquet


## 🎯 Seção 7 — TOFU Term Frequency

In [7]:

print("\n🎯 TOFU Term Frequency")

# Filtrar global_wc pelos termos TOFU
df_tofu = df_global_wc[df_global_wc["term"].isin(TOFU_TERMS)].copy()
df_tofu = df_tofu.sort_values("count", ascending=False).reset_index(drop=True)
df_tofu["tofu_rank"] = range(1, len(df_tofu) + 1)

# Enriquecer com presença por fonte
_tofu_src = (
    df_src_wc[df_src_wc["term"].isin(TOFU_TERMS)]
    .groupby("term")
    .agg(n_sources=("source_label", "nunique"))
    .reset_index()
)
df_tofu = df_tofu.merge(_tofu_src, on="term", how="left")
df_tofu["n_sources"] = df_tofu["n_sources"].fillna(0).astype(int)

_path = GOLD_WC_DIR / "tofu_frequency.parquet"
df_tofu.to_parquet(_path, index=False, compression="snappy")
print(f"  TOFU terms encontrados: {len(df_tofu)}/{len(TOFU_TERMS)}")
print(df_tofu.head(15).to_string(index=False))
print(f"\n✅ Salvo: {_path}")


🎯 TOFU Term Frequency
  TOFU terms encontrados: 29/32
        term  count  rank  tofu_rank  n_sources
       fundo    189     3          1          9
      fundos    183     5          2         11
       renda    183     7          3         10
  dividendos    170    10          4          8
        fiis    152    17          5         11
imobiliarios    110    22          6         10
investimento    103    30          7          9
       papel     74    58          8         10
      gestao     69    65          9          3
 imobiliario     61    81         10          8
    liquidez     60    88         11          5
   dividendo     59    91         12          6
       cotas     48   127         13          5
   portfolio     47   144         14          3
      tijolo     43   163         15          5

✅ Salvo: /Users/fabicampanari/Desktop/project-fii-marketing-intelligence-platform/data/gold/word_count/tofu_frequency.parquet



## ⚠️ Seção 8 — Negative Context Analysis

**Desafio do acaademico:** _"identificar palavras de alta frequência em contexto negativo"._

Implementação: para cada termo de risco, conta co-ocorrências com palavras negativas
dentro de uma janela de ±5 palavras no mesmo documento.

In [8]:

print("\n⚠️  Negative Context Analysis")

NEGATIVE_WORDS = {
    "queda", "reducao", "corte", "risco", "crise", "perda", "negativo",
    "prejuizo", "calote", "default", "inadimplencia", "vacancia",
    "desvalorizacao", "baixo", "fraco", "critica", "problema",
    "preocupante", "alerta", "cautela",
}

def negative_ctx_ratio(text: str, term: str, window: int = 5) -> float:
    tokens = tokenize(text)
    if term not in tokens: return 0.0
    positions = [i for i, t in enumerate(tokens) if t == term]
    neg_windows = 0
    for pos in positions:
        win = set(tokens[max(0, pos - window): pos + window + 1])
        if win & NEGATIVE_WORDS:
            neg_windows += 1
    return neg_windows / len(positions) if positions else 0.0

# RDD: (term, total_count, negative_ctx_count, neg_ratio, risk_level)
texts_list  = df_silver["text_clean"].tolist()

neg_ctx_rdd = spark.sparkContext.parallelize(list(NEGATIVE_CONTEXT_TERMS.keys()))

neg_results = []
for term in NEGATIVE_CONTEXT_TERMS.keys():
    total_count = df_global_wc[df_global_wc["term"] == term]["count"].sum() if term in df_global_wc["term"].values else 0
    ratios = [negative_ctx_ratio(t, term) for t in texts_list if term in t]
    avg_ratio = float(np.mean(ratios)) if ratios else 0.0
    neg_results.append({
        "term": term,
        "global_count": int(total_count),
        "negative_ctx_ratio": round(avg_ratio, 4),
        "risk_level": NEGATIVE_CONTEXT_TERMS[term],
        "n_docs_with_term": len(ratios),
    })

df_neg_ctx = pd.DataFrame(neg_results).sort_values("negative_ctx_ratio", ascending=False)

_path = GOLD_WC_DIR / "negative_context.parquet"
df_neg_ctx.to_parquet(_path, index=False, compression="snappy")
print(df_neg_ctx.to_string(index=False))
print(f"\n✅ Salvo: {_path}")


⚠️  Negative Context Analysis
          term  global_count  negative_ctx_ratio risk_level  n_docs_with_term
         fraco             2              1.0000        low                 2
        calote             3              1.0000       high                 1
       default             3              1.0000       high                 3
         risco           158              0.9143     medium                35
         queda            47              0.8966     medium                29
      negativo             4              0.8000       high                 5
         crise             2              0.6667       high                 3
         corte            10              0.4667     medium                15
         baixo            46              0.1818        low                33
  volatilidade            25              0.1538        low                13
         venda            39              0.1056        low                20
         perda             3     

## 📋 Seção 9 — Resumo de Outputs

In [9]:

print("\n📋 OUTPUTS NB03 — Gold/word_count/")
print("=" * 60)
for fname, desc in [
    ("global_word_count.parquet", "Frequência global de todos os termos"),
    ("source_word_count.parquet", "Frequência por fonte (source_label × term)"),
    ("tofu_frequency.parquet",    "Termos TOFU rankeados por frequência + n_sources"),
    ("negative_context.parquet", "Termos de risco com ratio de contexto negativo"),
]:
    p = GOLD_WC_DIR / fname
    if p.exists():
        size = p.stat().st_size // 1024
        print(f"  ✅ {fname:<42} {size:>5} KB — {desc}")
    else:
        print(f"  ❌ {fname} — não gerado")


📋 OUTPUTS NB03 — Gold/word_count/
  ✅ global_word_count.parquet                    112 KB — Frequência global de todos os termos
  ✅ source_word_count.parquet                     78 KB — Frequência por fonte (source_label × term)
  ✅ tofu_frequency.parquet                         4 KB — Termos TOFU rankeados por frequência + n_sources
  ✅ negative_context.parquet                       3 KB — Termos de risco com ratio de contexto negativo



## ✅ NB03 Completo

| Artefato | Colunas-chave | Consumido por |
|----------|---------------|---------------|
| `global_word_count.parquet` | term, count, rank | NB04, NB05 |
| `source_word_count.parquet` | source_label, term, count | NB04, NB06 |
| `tofu_frequency.parquet` | term, count, n_sources, tofu_rank | NB04, NB05, NB06 |
| `negative_context.parquet` | term, neg_ctx_ratio, risk_level | NB04, NB05, NB06 |

▶️ **Próximo:** `04_tfidf_bm25.ipynb`